In [ ]:
import sys, subprocess
def ensure_package(pkg_name, import_name=None):
    import_name = import_name or pkg_name
    try:
        __import__(import_name)
        print(f'{pkg_name} already installed.')
    except ImportError:
        print(f'Installing {pkg_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_name])
ensure_package('imbalanced-learn', 'imblearn')
ensure_package('xgboost', 'xgboost')


In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.metrics import roc_curve, precision_recall_curve
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
print('All libraries imported!')


In [ ]:
df = pd.read_csv('Dataset.csv', low_memory=False)
if 'Unnamed: 0' in df.columns: df = df.drop(columns=['Unnamed: 0'])
assert 'Patient_ID' in df.columns and 'SepsisLabel' in df.columns
print(f'Shape: {df.shape}')

In [ ]:
ignore_cols = ['SepsisLabel', 'Patient_ID', 'Unit1', 'Unit2']
feature_cols = [c for c in df.columns if c not in ignore_cols]
X = df[feature_cols].copy()
y = df['SepsisLabel'].values
groups = df['Patient_ID'].values
print(f'Features: {len(feature_cols)}')

In [ ]:
def fit_winsorization_bounds(X_train, features, k=1.5):
    bounds = {}
    for col in features:
        Q1 = X_train[col].quantile(0.25)
        Q3 = X_train[col].quantile(0.75)
        IQR = Q3 - Q1
        bounds[col] = (Q1 - k*IQR, Q3 + k*IQR)
    return bounds

def apply_winsorization(X_df, bounds):
    X_c = X_df.copy()
    for col, (lo, hi) in bounds.items():
        if col in X_c.columns:
            X_c[col] = X_c[col].clip(lower=lo, upper=hi)
    return X_c

In [ ]:
def evaluate_model_groupkfold(model_name, model_inst, X, y, groups, n_splits=5, random_state=42):
    gkf = GroupKFold(n_splits=n_splits)
    X = X.reset_index(drop=True)
    y = np.asarray(y)
    groups = np.asarray(groups)
    fold_metrics, oof_true, oof_prob, oof_pred = [], [], [], []
    print(f'=== {model_name} | {n_splits}-Fold GroupKFold ===')
    for fold, (tr, va) in enumerate(gkf.split(X, y, groups), 1):
        X_tr, y_tr = X.iloc[tr].copy(), y[tr]
        X_va, y_va = X.iloc[va].copy(), y[va]
        print(f'  Fold {fold}: train_pos={int(y_tr.sum())}, val_pos={int(y_va.sum())}')
        bounds = fit_winsorization_bounds(X_tr, feature_cols)
        X_tr = apply_winsorization(X_tr, bounds)
        X_va = apply_winsorization(X_va, bounds)
        imp = SimpleImputer(strategy='median')
        X_tr_i = imp.fit_transform(X_tr)
        X_va_i = imp.transform(X_va)
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr_i)
        X_va_s = sc.transform(X_va_i)
        n_pos = int(y_tr.sum())
        if n_pos < 6:
            X_tr_r, y_tr_r = X_tr_s, y_tr
        else:
            smote = SMOTE(random_state=random_state, k_neighbors=min(5, n_pos-1))
            X_tr_r, y_tr_r = smote.fit_resample(X_tr_s, y_tr)
        model_inst.fit(X_tr_r, y_tr_r)
        prob = model_inst.predict_proba(X_va_s)[:,1]
        pred = (prob >= 0.5).astype(int)
        fold_metrics.append({'fold': fold, 'Accuracy': accuracy_score(y_va,pred),
            'Precision': precision_score(y_va,pred,zero_division=0),
            'Recall': recall_score(y_va,pred,zero_division=0),
            'F1-Score': f1_score(y_va,pred,zero_division=0),
            'AUROC': roc_auc_score(y_va,prob),
            'AUPRC': average_precision_score(y_va,prob)})
        oof_true.extend(y_va.tolist()); oof_prob.extend(prob.tolist()); oof_pred.extend(pred.tolist())
        print(f'    AUROC={fold_metrics[-1]["AUROC"]:.4f} | AUPRC={fold_metrics[-1]["AUPRC"]:.4f} | Recall={fold_metrics[-1]["Recall"]:.4f}')
    mdf = pd.DataFrame(fold_metrics)
    mean_r = mdf.drop(columns=['fold']).mean().to_dict()
    std_r = mdf.drop(columns=['fold']).std().to_dict()
    print(f'  MEAN AUROC: {mean_r["AUROC"]:.4f} +/- {std_r["AUROC"]:.4f}')
    print(f'  MEAN Recall: {mean_r["Recall"]:.4f} +/- {std_r["Recall"]:.4f}')
    return {'metrics_df': mdf, 'mean': mean_r, 'std': std_r,
            'oof_true': np.array(oof_true), 'oof_proba': np.array(oof_prob), 'oof_pred': np.array(oof_pred)}

In [ ]:
lr_results = evaluate_model_groupkfold('Logistic Regression (SMOTE)',
    LogisticRegression(max_iter=500, random_state=42), X, y, groups)

In [ ]:
rf_results = evaluate_model_groupkfold('Random Forest (SMOTE)',
    RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1), X, y, groups)

In [ ]:
hgb_results = evaluate_model_groupkfold('HistGradientBoosting (SMOTE)',
    HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=5, random_state=42), X, y, groups)

In [ ]:
models_dict = {'Logistic Regression (SMOTE)': lr_results,
               'Random Forest (SMOTE)': rf_results,
               'HistGradientBoosting (SMOTE)': hgb_results}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for name, res in models_dict.items():
    fpr, tpr, _ = roc_curve(res['oof_true'], res['oof_proba'])
    auc_val = roc_auc_score(res['oof_true'], res['oof_proba'])
    axes[0].plot(fpr, tpr, label=f'{name} (AUROC={auc_val:.3f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0].set_title('Baseline ROC Curves')
axes[0].legend(loc='lower right')
for name, res in models_dict.items():
    prec, rec, _ = precision_recall_curve(res['oof_true'], res['oof_proba'])
    auprc_val = average_precision_score(res['oof_true'], res['oof_proba'])
    axes[1].plot(rec, prec, label=f'{name} (AUPRC={auprc_val:.3f})')
axes[1].set_title('Baseline PR Curves')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.savefig('baseline_roc_pr_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, models_dict.items()):
    cm = confusion_matrix(res['oof_true'], res['oof_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig('baseline_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def get_metric(d, *names, default=np.nan):
    for n in names:
        if isinstance(d, dict) and n in d:
            try: return float(d[n])
            except: continue
    return default

def fmt(v):
    return f'{v:.4f}' if isinstance(v,(int,float)) and not np.isnan(v) else 'N/A'

rows = []
for name, res in models_dict.items():
    m = res['mean']
    rows.append({'Model Architecture': name,
                 'AUROC': fmt(m['AUROC']), 'AUPRC': fmt(m['AUPRC']),
                 'Recall': fmt(m['Recall']), 'Precision': fmt(m['Precision']),
                 'F1': fmt(m['F1-Score']), 'Accuracy': fmt(m['Accuracy'])})

if os.path.exists('paper_metrics.json'):
    with open('paper_metrics.json') as f: paper_data = json.load(f)
    for key, label in [('xgb_6h','Advanced XGBoost (6h)'),
                       ('xgb_12h','Advanced XGBoost (12h)')]:
        block = paper_data.get(key)
        if not isinstance(block, dict): continue
        rows.append({'Model Architecture': label,
            'AUROC': fmt(get_metric(block,'AUROC','roc_auc','auroc')),
            'AUPRC': fmt(get_metric(block,'AUPRC','pr_auc','auprc')),
            'Recall': fmt(get_metric(block,'Recall','recall')),
            'Precision': fmt(get_metric(block,'Precision','precision')),
            'F1': fmt(get_metric(block,'F1-Score','f1','f1_score')),
            'Accuracy': fmt(get_metric(block,'Accuracy','accuracy','acc'))})

comp_df = pd.DataFrame(rows)
comp_df.to_csv('baseline_comparison_results.csv', index=False)
comp_df

In [ ]:
summary = {'logistic_regression': lr_results['mean'],
           'random_forest': rf_results['mean'],
           'hist_gradient_boosting': hgb_results['mean']}
with open('baseline_metrics.json','w') as f: json.dump(summary, f, indent=2)
lr_results['metrics_df'].to_csv('baseline_logreg_per_fold.csv', index=False)
rf_results['metrics_df'].to_csv('baseline_rf_per_fold.csv', index=False)
hgb_results['metrics_df'].to_csv('baseline_hgb_per_fold.csv', index=False)
print('Saved all baseline artifacts!')